# LangGraph 010 — A Chatbot with Conversation Memory

One node, and a state that is the conversation. Needs `langgraph`; **no API
key** — the model is scripted so the memory being measured is LangGraph's.

| Part | What we check |
|---|---|
| A | the state, and why `BaseMessage` plus a reducer |
| B | no checkpointer: **2 messages** on every turn |
| C | a checkpointer and a thread id: **2, 4, 6, 8** |
| D | two threads that cannot see each other |
| E | what `add_messages` does that `operator.add` cannot |
| F | what each turn is handed — and trimming it |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The state

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# Two decisions in one line.
#
# BaseMessage, not str: you talk to a chat model in messages - HumanMessage,
# AIMessage, SystemMessage, ToolMessage - and all four inherit from
# BaseMessage, so the list can hold any of them.
#
# add_messages, not nothing: without a reducer each turn would REPLACE the
# conversation instead of extending it. add_messages is operator.add's
# specialised cousin for messages, and the next block shows what it adds.

## Part B — The problem

In [ ]:
import re
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

# A scripted model, so this runs with no API key. It answers only from the
# messages it is handed - which is the whole point of what follows.
def scripted_llm(messages):
    text = messages[-1].content.lower()
    history = " ".join(m.content for m in messages[:-1])
    if "my name is" in text:
        name = messages[-1].content.split("my name is")[-1].strip(" .!?")
        return AIMessage(content=f"Hello {name}, nice to meet you.")
    if "my name" in text:
        found = re.search(r"my name is ([A-Z][a-z]+)", history)
        return AIMessage(content=f"Your name is {found.group(1)}." if found
                         else "I do not have access to that.")
    return AIMessage(content="Noted.")

def chat_node(state: ChatState) -> dict:
    return {"messages": [scripted_llm(state["messages"])]}

graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)
chatbot = graph.compile()

for turn in ("Hi, my name is Ayesha", "What is my name?"):
    out = chatbot.invoke({"messages": [HumanMessage(content=turn)]})
    print(f"you: {turn}\nbot: {out['messages'][-1].content}  "
          f"({len(out['messages'])} messages in state)")

# you: Hi, my name is Ayesha
# bot: Hello Ayesha, nice to meet you.  (2 messages in state)
# you: What is my name?
# bot: I do not have access to that.  (2 messages in state)
#
# Two messages both times. The second invoke started from an empty state: at
# END the state is gone. Nothing is wrong with the graph or the model.

In [ ]:
# four turns, none of which can see the others
TURNS = ["Hi, my name is Ayesha", "What is my name?", "Add 10 to 100",
         "Now add 15 to the result"]
counts = []
for turn in TURNS:
    out = chatbot.invoke({"messages": [HumanMessage(content=turn)]})
    counts.append(len(out["messages"]))
    print(f"{turn:<26} -> {out['messages'][-1].content}")
print(counts)
assert counts == [2, 2, 2, 2]
print("one in, one out, every time - the state is erased at END")

## Part C — Persistence in two lines

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

chatbot = graph.compile(checkpointer=InMemorySaver())   # attached at COMPILE time
config = {"configurable": {"thread_id": "1"}}           # passed at INVOKE time

for turn in ("Hi, my name is Ayesha", "What is my name?"):
    out = chatbot.invoke({"messages": [HumanMessage(content=turn)]}, config=config)
    print(f"you: {turn}\nbot: {out['messages'][-1].content}  "
          f"({len(out['messages'])} messages in state)")

print(chatbot.get_state(config).values["messages"][0].content)

# you: Hi, my name is Ayesha
# bot: Hello Ayesha, nice to meet you.  (2 messages in state)
# you: What is my name?
# bot: Your name is Ayesha.  (4 messages in state)
# Hi, my name is Ayesha
#
# Same graph, same scripted model, two extra lines. The checkpointer saves the
# state at the end of each run and loads it at the start of the next, and the
# thread_id says WHICH conversation to load. Give a second user thread_id "2"
# and they see none of this.

In [ ]:
bot = graph.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "1"}}
counts = []
for turn in TURNS:
    out = bot.invoke({"messages": [HumanMessage(content=turn)]}, config=config)
    counts.append(len(out["messages"]))
    print(f"{turn:<26} -> {out['messages'][-1].content}")
print(counts)
assert counts == [2, 4, 6, 8]
assert "Ayesha" in out["messages"][3].content
print()
print("The scripted model did not change. What it was handed did.")
print(f"saved state: {len(bot.get_state(config).values['messages'])} messages, "
      f"{len(list(bot.get_state_history(config)))} checkpoints")

## Part D — Threads

In [ ]:
bot = graph.compile(checkpointer=InMemorySaver())
one = {"configurable": {"thread_id": "1"}}
two = {"configurable": {"thread_id": "2"}}

bot.invoke({"messages": [HumanMessage(content="Hi, my name is Ayesha")]}, config=one)
a = bot.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=one)
b = bot.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=two)
print("thread 1:", a["messages"][-1].content)
print("thread 2:", b["messages"][-1].content)
assert "Ayesha" in a["messages"][-1].content
assert "Ayesha" not in b["messages"][-1].content

# and the thread id is a key, not the memory
fresh = graph.compile(checkpointer=InMemorySaver())      # as if after a restart
after = fresh.invoke({"messages": [HumanMessage(content="What is my name?")]},
                     config=one)
print("new store, same thread id:", after["messages"][-1].content)
assert "Ayesha" not in after["messages"][-1].content
print("RAM does not survive a restart - that is what a database is for")

## Part E — `add_messages` against `operator.add`

In [ ]:
import operator
from langchain_core.messages import RemoveMessage
from langgraph.checkpoint.memory import MemorySaver

# (a) every message comes back with an id
merged = add_messages([HumanMessage(content="hi")], [AIMessage(content="yo")])
print([bool(m.id) for m in merged])

# (b) the same id twice - append, or replace?
first = HumanMessage(content="first draft", id="m1")
second = HumanMessage(content="corrected", id="m1")
print("add_messages:", [m.content for m in add_messages([first], [second])])
print("operator.add:", [m.content for m in operator.add([first], [second])])

# (c) a bare string is coerced
print(type(add_messages([], ["hello"])[0]).__name__)

# (d) and a message can be deleted
print(add_messages([first], [RemoveMessage(id="m1")]))

# (e) the two names for one class
print("MemorySaver is InMemorySaver:", MemorySaver is InMemorySaver)

# [True, True]
# add_messages: ['corrected']
# operator.add: ['first draft', 'corrected']
# HumanMessage
# []
# MemorySaver is InMemorySaver: True
#
# So add_messages is not just "append for messages". It assigns ids, REPLACES
# by id, accepts a plain string, and honours RemoveMessage - which is how a
# message gets edited or dropped later in this track. operator.add can do none
# of that: it would have kept both drafts.

In [ ]:
# replacement by id is how a long tool result becomes a short summary
long_result = AIMessage(content="x" * 4000, id="tool-1")
summary = AIMessage(content="[4,000 characters of search results, summarised]",
                    id="tool-1")
before = [HumanMessage(content="search for trains"), long_result]
after = add_messages(before, [summary])
print(f"{sum(len(m.content) for m in before):,} chars -> "
      f"{sum(len(m.content) for m in after):,} chars, "
      f"{len(before)} messages -> {len(after)}")
assert len(after) == 2 and len(after[1].content) < 100
print("same message count, 98% less text - operator.add could not do this")

## Part F — What memory costs, and trimming it

In [ ]:
# Memory is re-sent, not remembered on the far side. Measure what the node is
# handed on each of ten turns.
handed = []

def chat_node(state: ChatState) -> dict:
    handed.append(sum(len(m.content) for m in state["messages"]))
    return {"messages": [scripted_llm(state["messages"])]}

graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)
bot = graph.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "cost"}}

for i in range(1, 11):
    bot.invoke({"messages": [HumanMessage(content=f"Turn {i}: tell me something.")]},
               config=config)

print(handed)
print(f"turn 10 was handed {handed[-1] / handed[0]:.1f}x turn 1")
print(f"total {sum(handed):,} characters against "
      f"{handed[0] * 10:,} for ten independent turns")

# [26, 58, 90, 122, 154, 186, 218, 250, 282, 315]
# turn 10 was handed 12.1x turn 1
# total 1,701 characters against 260 for ten independent turns
#
# The history grows by a constant each turn, so the TOTAL grows with the square
# of the conversation length. That is why trimming, windowing and summarising
# exist - and why lesson 011's persistence lesson matters for the bill as well
# as for the memory.

In [ ]:
# keep only the last 4 messages, with RemoveMessage
from langchain_core.messages import RemoveMessage

def trimming_chat_node(state: ChatState) -> dict:
    reply = scripted_llm(state["messages"])
    updates = [reply]
    keep = 4
    older = state["messages"][:-(keep - 1)] if len(state["messages"]) >= keep else []
    updates += [RemoveMessage(id=m.id) for m in older]
    return {"messages": updates}

trimmed = StateGraph(ChatState)
trimmed.add_node("chat_node", trimming_chat_node)
trimmed.add_edge(START, "chat_node")
trimmed.add_edge("chat_node", END)
tbot = trimmed.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "trim"}}

sizes = []
for i in range(1, 11):
    out = tbot.invoke({"messages": [HumanMessage(content=f"Turn {i}: hello.")]},
                      config=cfg)
    sizes.append(len(out["messages"]))
print(sizes)
assert max(sizes) <= 4
print()
print("A sliding window holds the cost flat instead of letting it grow with the")
print("square of the conversation. The price is that the bot forgets - which is")
print("why summarising the old messages beats dropping them.")

## What this notebook established

- A chatbot is a one-node sequential workflow; the conversation loop is in your
  program.
- With no checkpointer, four turns each held **2 messages** — the state is
  erased at END, and *the result* in turn 4 had no referent.
- A checkpointer plus a `thread_id` gave **2, 4, 6, 8**, with the same scripted
  model answering what it had just failed.
- Threads are separate, and a fresh in-memory store forgets everything even on
  the same thread id.
- `add_messages` assigns ids, **replaces by id**, coerces a plain string and
  honours `RemoveMessage`. Replacement by id turned a 4,000-character tool
  result into a 47-character summary without changing the message count.
- Each turn is handed the whole history: turn 10 got **12.1x** turn 1. A
  four-message window holds that flat.

## Exercises

1. Replace the sliding window with a summary: when the history passes 8
   messages, collapse the oldest into one `SystemMessage`.
2. Give two threads the same first message and confirm the checkpoints are
   independent in `get_state_history`.
3. Measure the cost curve with and without trimming over 50 turns. Where do the
   two lines cross?
4. Write a `SystemMessage` at the start of a thread and check it is still there
   after ten turns — then after trimming.